# LSTM vs Baseline Comparison

Notebook ini membandingkan checkpoint LSTM aktif dengan baseline model pada test split yang sama. Karena workflow ini memuat model deep learning, seluruh eksekusi disimpan sebagai `.ipynb`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("C:/AIproject/AWAI")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

In [ ]:
from datetime import datetime
import json

import numpy as np
import pandas as pd
import torch

from traffic_prediction.config.settings import load_config
from traffic_prediction.data.processor import DataProcessor
from traffic_prediction.evaluation.baselines import (
    BaselineEvaluator,
    HistoricalAverageBaseline,
    LinearRegressionBaseline,
    PersistenceBaseline,
)
from traffic_prediction.evaluation.metrics import calculate_regression_metrics
from traffic_prediction.features.offline import FeatureEngineer
from traffic_prediction.features.spatial import build_neighbor_mapping
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM
from traffic_prediction.models.registry import ModelRegistry
from traffic_prediction.pipelines.offline import select_feature_columns

config = load_config(project_root=PROJECT_ROOT)
run_stamp = datetime.now().strftime("baseline-comparison-%Y%m%d-%H%M%S")
report_dir = config.paths.reports_dir / run_stamp
report_dir.mkdir(parents=True, exist_ok=True)

registry = ModelRegistry(config.paths.models_dir / "registry.json")
active_model = registry.get_active()
if active_model is None or active_model.model_type != "lstm":
    raise RuntimeError("Active model must be a trained LSTM before running this notebook")

model_dir = Path(active_model.artifact_path)
model_config_payload = json.loads((model_dir / "model_config.json").read_text(encoding="utf-8"))
print("Comparison run:", run_stamp)
print("Active model:", active_model.model_version)
print("Model dir:", model_dir)

## Rebuild the Same Data Contract

The comparison rebuilds preprocessing deterministically so baselines and LSTM use the same chronological split, feature manifest, scaler, lookback, and horizon.

In [ ]:
processor = DataProcessor(config.data, config.features)
traffic_raw, validation_report = processor.load_traffic_csv(config.paths.traffic_csv)
roads = processor.load_roads_csv(config.paths.roads_csv)
cleaned = processor.clean(traffic_raw)

neighbor_mapping = build_neighbor_mapping(
    roads,
    neighbor_count=config.features.spatial_neighbor_count,
)
feature_engineer = FeatureEngineer(
    neighbor_mapping=neighbor_mapping,
    lag_steps=config.features.lag_steps,
    rolling_windows=config.features.rolling_windows,
)
featured = feature_engineer.extract_features(cleaned)
feature_columns = select_feature_columns(featured)

train, validation, test, split_stats = processor.chronological_split(featured)
train_scaled = processor.fit_transform_train(train, feature_columns)
validation_scaled = processor.transform_eval(validation)
test_scaled = processor.transform_eval(test)

X_train, y_train, train_metadata = processor.create_sequences(train_scaled, feature_columns)
X_test, y_test, test_metadata = processor.create_sequences(test_scaled, feature_columns)

print("Feature count:", len(feature_columns))
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

In [ ]:
model_config = LSTMModelConfig(**model_config_payload["model_config"])
checkpoint = torch.load(model_config_payload["checkpoint_path"], map_location="cpu")
model = TrafficLSTM(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def predict_in_batches(X: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32)
            outputs.append(model(batch).numpy())
    return np.concatenate(outputs, axis=0)

lstm_predictions_scaled = predict_in_batches(X_test)
lstm_predictions_kmh = processor.scaler_store.inverse_transform_speed(
    lstm_predictions_scaled.reshape(-1, 1)
).reshape(lstm_predictions_scaled.shape)
y_test_kmh = processor.scaler_store.inverse_transform_speed(y_test.reshape(-1, 1)).reshape(y_test.shape)
lstm_metrics = calculate_regression_metrics(y_test_kmh, lstm_predictions_kmh)
lstm_metrics.to_dict()

In [ ]:
frequency = pd.Timedelta(config.data.frequency)
current_speed_index = feature_columns.index("current_speed")
baseline_feature_columns = [f"feature__{column}" for column in feature_columns]

def inverse_speed(values: np.ndarray) -> np.ndarray:
    return processor.scaler_store.inverse_transform_speed(values.reshape(-1, 1)).reshape(values.shape)

def build_horizon_frame(X: np.ndarray, y: np.ndarray, metadata) -> pd.DataFrame:
    y_kmh = inverse_speed(y)
    last_step_features = X[:, -1, :]
    lag_1_kmh = inverse_speed(X[:, -1, current_speed_index])
    rows = []
    for sequence_index, item in enumerate(metadata):
        for horizon_index in range(y.shape[1]):
            row = {
                "road_id": item.road_id,
                "timestamp": pd.Timestamp(item.target_start) + horizon_index * frequency,
                "horizon_minutes": int((horizon_index + 1) * frequency.total_seconds() / 60),
                "actual_speed": float(y_kmh[sequence_index, horizon_index, 0]),
                "lag_1": float(lag_1_kmh[sequence_index]),
            }
            for column, value in zip(baseline_feature_columns, last_step_features[sequence_index]):
                row[column] = float(value)
            rows.append(row)
    return pd.DataFrame(rows)

train_eval = build_horizon_frame(X_train, y_train, train_metadata)
test_eval = build_horizon_frame(X_test, y_test, test_metadata)

print("Train baseline rows:", len(train_eval))
print("Test baseline rows:", len(test_eval))

In [ ]:
baseline_evaluator = BaselineEvaluator(
    baselines=[
        PersistenceBaseline(),
        HistoricalAverageBaseline(),
        LinearRegressionBaseline(feature_columns=baseline_feature_columns),
    ]
)
baseline_comparison = baseline_evaluator.evaluate(train_eval, test_eval)
baseline_table = baseline_comparison.to_dataframe()

lstm_row = pd.DataFrame([
    {
        "model": active_model.model_version,
        "status": "ok",
        "mae": lstm_metrics.mae,
        "rmse": lstm_metrics.rmse,
        "mape": lstm_metrics.mape,
        "accuracy_from_mape": 100.0 - lstm_metrics.mape,
        "r2": lstm_metrics.r2,
        "sample_count": lstm_metrics.sample_count,
        "detail": "trained LSTM checkpoint",
    }
])
baseline_table["accuracy_from_mape"] = 100.0 - baseline_table["mape"]
comparison_table = pd.concat([lstm_row, baseline_table], ignore_index=True)
comparison_table = comparison_table.sort_values(["status", "rmse", "mae"]).reset_index(drop=True)
comparison_table

In [ ]:
best_model = comparison_table.iloc[0].to_dict()
report = {
    "run_id": run_stamp,
    "active_model_version": active_model.model_version,
    "created_at": datetime.now().isoformat(),
    "split_statistics": split_stats.__dict__,
    "sequence_shapes": {
        "X_train": list(X_train.shape),
        "y_train": list(y_train.shape),
        "X_test": list(X_test.shape),
        "y_test": list(y_test.shape),
    },
    "best_model_by_rmse": best_model,
    "comparison": comparison_table.to_dict(orient="records"),
}

comparison_csv = report_dir / "baseline_comparison.csv"
comparison_json = report_dir / "baseline_comparison.json"
comparison_table.to_csv(comparison_csv, index=False)
comparison_json.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

print("Best model by RMSE:", best_model["model"])
print("CSV:", comparison_csv)
print("JSON:", comparison_json)
comparison_table